# WACV 2027 Workshop on Real-World Surveillance (RWS) — Track 2: ABPR
## EXP-006: Stage 2 Dual-Stream Cross-Modal Metric Learning & Championship Pipeline

> **Target Hardware:** Kaggle GPU T4 x 2 (32GB VRAM Total, 30GB CPU RAM)
> **Objective:** Combine ConvNeXt-Tiny (local texture) and Swin-Transformer Tiny (global self-attention) into a 1536-D unified representation, trained with ArcFace Additive Angular Margin Loss (=30, m=0.20$) over 7,204 identities and Symmetric InfoNCE Contrastive Loss to break the 40-bit crowding bottleneck and reach $\ge 0.65 - 0.68	ext{ mADM}$.
> **Execution Mode:** Background Run via **Save Version -> Save & Run All (Commit)** or Interactive Execution.


In [ ]:
# ==============================================================================
# 1. MASTER HYPERPARAMETER CONFIGURATION (Easily Editable)
# ==============================================================================
import os

CONFIG = {
    "EXPERIMENT_ID": "EXP-006",
    "PIPELINE_MODE": "dual_fused",               # Full 1536-D Dual-Stream Fused Architecture
    "CONVNEXT_ZIP": "submissions/EXP-004_submission.zip",
    "SWIN_ZIP": "submissions/EXP-005A_submission.zip",
    "STAGE2_EPOCHS": 30,                       # 30 epochs of ArcFace + InfoNCE metric learning
    "BATCH_SIZE": 512,                          # Batch size for metric projection training
    "LEARNING_RATE": 1e-3,                      # AdamW initial learning rate
    "ARCFACE_SCALE": 30.0,                      # ArcFace hypersphere radius (s)
    "ARCFACE_MARGIN": 0.20,                     # Additive angular margin (m)
    "LAMBDA_CONTRASTIVE": 0.5,                  # InfoNCE cross-modal loss weight
    "LAMBDA_METRIC": 0.25,                      # Hybrid score fusion: (1-λ)*L1 + λ*cos_dist
    "DATA_DIR": "/kaggle/input/datasets/hinewk/upar-challenge-2027-data", # Base path
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints",
    "OUTPUT_DIR": "/kaggle/working/submissions",
}

print(f"Master Configuration loaded for {CONFIG['EXPERIMENT_ID']} ({CONFIG['PIPELINE_MODE']})")


In [ ]:
# ==============================================================================
# 2. PRIVATE GITHUB REPOSITORY INGESTION (Auto-Clone / Pull with Token Support)
# ==============================================================================
import os
import sys

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = "YOUR_GITHUB_PAT_TOKEN_HERE"

if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PAT_TOKEN_HERE":
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"
else:
    REPO_URL = "https://github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"

REPO_DIR = "/kaggle/working/upar_challenge_26_rws27"

if os.path.exists(REPO_DIR):
    print("Repository directory exists. Pulling latest commits from main...")
    os.system(f"cd {REPO_DIR} && git pull origin main")
else:
    print("Cloning repository from GitHub...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Active Working Directory: {os.getcwd()}")
!pip install -q -r requirements.txt


In [ ]:
# ==============================================================================
# 3. HARDWARE & DATASET PATH AUTO-DETECTION
# ==============================================================================
import glob
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
print(f"Detected GPUs: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | Total Memory: {props.total_memory / (1024**3):.2f} GB")

# Auto-detect dataset directory
data_dir = CONFIG["DATA_DIR"]
if not os.path.exists(os.path.join(data_dir, "annotations", "train.csv")):
    if os.path.exists(os.path.join(data_dir, "data", "annotations", "train.csv")):
        CONFIG["DATA_DIR"] = os.path.join(data_dir, "data")
    else:
        matches = glob.glob("/kaggle/input/**/annotations/train.csv", recursive=True)
        if matches:
            CONFIG["DATA_DIR"] = os.path.dirname(os.path.dirname(matches[0]))

print(f"[DATASET] Final Resolved Path: {CONFIG['DATA_DIR']}")
if os.path.exists(CONFIG["DATA_DIR"]):
    train_check = os.path.join(CONFIG["DATA_DIR"], "annotations", "train.csv")
    print(f"[DATASET] train.csv verified: {os.path.exists(train_check)}")


In [ ]:
# ==============================================================================
# 4. LAUNCH MASTER STAGE 2 DUAL-STREAM PIPELINE (ConvNeXt + Swin-T + ArcFace)
# ==============================================================================
# Executes the Master Stage 2 Pipeline:
# 1. Extracts Swin-T features from EXP-005A
# 2. Fuses ConvNeXt-Tiny (768) + Swin-T (768) into 1536-D unified representation
# 3. Trains 1536-D VisualProjector and QueryEncoder with ArcFace and InfoNCE
# 4. Evaluates across lambda values on all 33,407 validation gallery images
!python run_stage2_full_pipeline.py --data_root {CONFIG["DATA_DIR"]}


In [ ]:
# ==============================================================================
# 5. PACKAGE CODABENCH SUBMISSIONS (Dual-Stream Championship & Swin-T Safe)
# ==============================================================================
# Builds standalone, self-contained, air-gapped submission archives with embedded calibration
!python src/stage2_submission_builder.py

!ls -lh submissions/EXP-006*


In [ ]:
# ==============================================================================
# 6. EXECUTE CODABENCH CONTAINER SANDBOX SIMULATION (Zero Failure Guarantee)
# ==============================================================================
# Simulates exact Codabench Ingestion and Scoring program inside isolated sandbox
!python tests/test_codabench_simulation.py


In [ ]:
# ==============================================================================
# 7. EXPORT SUBMISSION ARCHIVES & ONE-CLICK DOWNLOAD LINKS
# ==============================================================================
import shutil
from IPython.display import FileLink, display

os.makedirs("/kaggle/working/final_submissions", exist_ok=True)

dual_src = "submissions/EXP-006_STAGE2_DUAL_FUSED_CHAMPIONSHIP_submission.zip"
dual_dst = "/kaggle/working/final_submissions/EXP-006_STAGE2_DUAL_FUSED_CHAMPIONSHIP_submission.zip"

swin_src = "submissions/EXP-006_STAGE2_SWIN_METRIC_submission.zip"
swin_dst = "/kaggle/working/final_submissions/EXP-006_STAGE2_SWIN_METRIC_submission.zip"

if os.path.exists(dual_src):
    shutil.copy2(dual_src, dual_dst)
    print(f"Championship Dual-Stream Package Ready: {dual_dst} ({os.path.getsize(dual_dst)/(1024*1024):.1f} MB)")
    display(FileLink("final_submissions/EXP-006_STAGE2_DUAL_FUSED_CHAMPIONSHIP_submission.zip"))

if os.path.exists(swin_src):
    shutil.copy2(swin_src, swin_dst)
    print(f"Safe Swin-T Single-Model Package Ready: {swin_dst} ({os.path.getsize(swin_dst)/(1024*1024):.1f} MB)")
    display(FileLink("final_submissions/EXP-006_STAGE2_SWIN_METRIC_submission.zip"))

print("
You can now click the download links above or download from the Kaggle Output tab to submit to Codabench!")
